# Notebook 4: Transporte, Violencia Intrafamiliar y Conclusiones

**DataJam Edición 4 — Universidad Distrital Francisco José de Caldas**

## Objetivo
Analizar el rol del transporte como barrera educativa (Encuesta Distrital + Multipropósito), la violencia intrafamiliar como factor contextual, y consolidar las conclusiones finales del proyecto.

## Análisis en este notebook
1. Transporte: percepción (Enc. Distrital) + tiempo real (Multipropósito 2021)
2. Percepción económica → Reprobación (por UPL)
3. Violencia intrafamiliar y entorno escolar
4. Cadena causal propuesta
5. Conclusiones y recomendaciones

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

PROJECT_ROOT = Path(os.path.abspath('.')).resolve()
if '__vsc_ipynb_file__' in dir():
    PROJECT_ROOT = Path(__vsc_ipynb_file__).resolve().parent.parent
else:
    for _ in range(10):
        if (PROJECT_ROOT / 'requirements.txt').exists() and (PROJECT_ROOT / 'scripts').exists():
            break
        PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output'

# Cargar tabla integrada
integrado = pd.read_csv(OUTPUT_DIR / 'tabla_integrada_localidad.csv')
df_deser = pd.read_csv(OUTPUT_DIR / 'desercion_por_upl.csv')

LOC_NOMBRES = {
    '01': 'Usaquén', '02': 'Chapinero', '03': 'Santa Fe', '04': 'San Cristóbal',
    '05': 'Usme', '06': 'Tunjuelito', '07': 'Bosa', '08': 'Kennedy',
    '09': 'Fontibón', '10': 'Engativá', '11': 'Suba', '12': 'Barrios Unidos',
    '13': 'Teusaquillo', '14': 'Los Mártires', '15': 'Antonio Nariño',
    '16': 'Puente Aranda', '17': 'La Candelaria', '18': 'Rafael Uribe Uribe',
    '19': 'Ciudad Bolívar', '20': 'Sumapaz'
}

## 4.1 Transporte: Percepción (Encuesta Distrital 2025)

In [2]:
enc_path = DATA_DIR / 'encuesta_distrital' / 'base_ano_movil_2025.csv'
if enc_path.exists():
    df_enc = pd.read_csv(enc_path, low_memory=False)
    print(f"Encuesta Distrital: {df_enc.shape[0]} registros, {df_enc.shape[1]} variables")
    
    # Agregar por UPL: satisfacción con transporte y percepción de pobreza
    agg_enc = df_enc.groupby('Cod_UPL').agg(
        Sat_Transporte=('IPMIV_A', 'mean'),
        Pct_Pobre=('C303', lambda x: (x == 1).mean() * 100),
        Pct_Ing_Prec=('Ax502', lambda x: x.isin([1, 2]).mean() * 100),
    ).reset_index()
    
    # Cruzar con deserción
    deser_map = dict(zip(df_deser['Cod_UPL'], df_deser['Desercion_Of']))
    repr_map = dict(zip(df_deser['Cod_UPL'], df_deser['Reprobacion_Of']))
    agg_enc['Desercion'] = agg_enc['Cod_UPL'].map(deser_map)
    agg_enc['Reprobacion'] = agg_enc['Cod_UPL'].map(repr_map)
    agg_enc = agg_enc.dropna(subset=['Desercion'])
    
    r_sat_pob, p_sat_pob = stats.pearsonr(agg_enc['Sat_Transporte'], agg_enc['Pct_Pobre'])
    r_ing_repr, p_ing_repr = stats.pearsonr(agg_enc['Pct_Ing_Prec'], agg_enc['Reprobacion'])
    
    print(f"\nCorrelación Satisfacción transporte vs Pobreza: r={r_sat_pob:.3f}, p={p_sat_pob:.4f}")
    print(f"Correlación Ingresos precarios vs Reprobación: r={r_ing_repr:.3f}, p={p_ing_repr:.4f}")
else:
    print('⚠ Encuesta Distrital no disponible. Descárgala manualmente desde la SDP.')
    agg_enc = pd.DataFrame()

Encuesta Distrital: 13082 registros, 287 variables

Correlación Satisfacción transporte vs Pobreza: r=-0.636, p=0.0002
Correlación Ingresos precarios vs Reprobación: r=-0.470, p=0.0087


## 4.2 Transporte: Tiempo real al colegio (Multipropósito 2021)

In [3]:
tiempo_path = DATA_DIR / 'encuesta_multiproposito' / 'em2021_tiempo_colegio.csv'
if tiempo_path.exists():
    est = pd.read_csv(tiempo_path)
    est = est[est['Minutos_al_Colegio'] < 90]
    print(f"Estudiantes con datos de tiempo: {len(est)}")
    
    # Tiempo por condición económica
    print("\nTiempo promedio al colegio por nivel de ingresos:")
    for ing, label in [(1, 'No alcanzan'), (2, 'Solo mínimos'), (3, 'Pueden ahorrar')]:
        sub = est[est['Suficiencia_Ingresos'] == ing]
        if len(sub) > 30:
            prom = np.average(sub['Minutos_al_Colegio'], weights=sub['Factor_Expansion'])
            pct_30 = np.average((sub['Minutos_al_Colegio'] > 30).astype(int),
                               weights=sub['Factor_Expansion']) * 100
            print(f"  {label:15s}: {prom:.0f} min (>{30}min: {pct_30:.0f}%)")
    
    # Tiempo por localidad
    tiempo_loc = est.groupby('NOMBRE_LOCALIDAD').apply(
        lambda g: np.average(g['Minutos_al_Colegio'], weights=g['Factor_Expansion']),
        include_groups=False
    ).reset_index()
    tiempo_loc.columns = ['Localidad', 'Minutos_Colegio']
    tiempo_loc = tiempo_loc.sort_values('Minutos_Colegio', ascending=False)
    
    print("\nTiempo al colegio por localidad (top 10):")
    for _, row in tiempo_loc.head(10).iterrows():
        print(f"  {row['Localidad']:20s}: {row['Minutos_Colegio']:.0f} min")
else:
    print('⚠ Datos de tiempo al colegio no disponibles.')
    print('  Ejecuta: python scripts/descargar_datos.py --dataset encuesta_multiproposito --procesar')

Estudiantes con datos de tiempo: 7219

Tiempo promedio al colegio por nivel de ingresos:
  No alcanzan    : 21 min (>30min: 15%)
  Solo mínimos   : 24 min (>30min: 21%)
  Pueden ahorrar : 26 min (>30min: 24%)

Tiempo al colegio por localidad (top 10):
  La Candelaria       : 48 min
  Tunjuelito          : 46 min
  Barrios Unidos      : 45 min
  Usme                : 44 min
  Ciudad Bolívar      : 40 min
  Santa Fe            : 40 min
  Puente Aranda       : 40 min
  Bosa                : 39 min
  Engativá            : 39 min
  Rafael Uribe Uribe  : 38 min


## 4.3 Violencia Intrafamiliar en menores

In [4]:
vif_path = DATA_DIR / 'violencia_intrafamiliar' / 'osb_saludmental-vintrafamiliar.csv'
if vif_path.exists():
    df_vif = pd.read_csv(vif_path, sep=';', encoding='utf-8-sig',
                         usecols=['ano', 'grupoedad', 'NOMBRE_LOCALIDAD'],
                         low_memory=False)
    
    # Filtrar menores en edad escolar 2020-2025
    menores = df_vif[
        (df_vif['grupoedad'].isin(['De 1 - 5 años', 'De 6 - 13 años', 'De 14 - 17 años'])) &
        (df_vif['ano'].between(2020, 2025))
    ]
    
    # Casos por localidad
    vif_loc = menores.groupby('NOMBRE_LOCALIDAD').size().reset_index(name='Casos_VIF')
    
    # Cruzar con matrícula para tasa
    if 'Matricula' in integrado.columns:
        vif_loc = vif_loc.merge(
            integrado[['Localidad', 'Matricula', 'Pobreza_Monetaria', 'Reprobacion_Of']],
            left_on='NOMBRE_LOCALIDAD', right_on='Localidad', how='left'
        )
        vif_loc['Tasa_VIF_x1000'] = vif_loc['Casos_VIF'] / vif_loc['Matricula'] * 1000
        vif_loc = vif_loc.dropna(subset=['Reprobacion_Of', 'Tasa_VIF_x1000'])
        
        if len(vif_loc) > 5:
            r_vif, p_vif = stats.pearsonr(vif_loc['Tasa_VIF_x1000'], vif_loc['Reprobacion_Of'])
            print(f"Correlación VIF vs Reprobación: r={r_vif:.3f}, p={p_vif:.4f}")
    
    print(f"\nTotal casos VIF en menores (2020-2025): {len(menores):,}")
    print("\nTop 5 localidades:")
    print(vif_loc.nlargest(5, 'Casos_VIF')[['NOMBRE_LOCALIDAD', 'Casos_VIF']].to_string(index=False))
else:
    print('⚠ Datos de violencia intrafamiliar no disponibles.')

Correlación VIF vs Reprobación: r=-0.078, p=0.7664

Total casos VIF en menores (2020-2025): 148,929

Top 5 localidades:
NOMBRE_LOCALIDAD  Casos_VIF
  Ciudad Bolívar      22685
         Kennedy      20712
            Bosa      19209
            Suba      17037
            Usme      12734


## 4.4 Cadena causal propuesta

Con base en las correlaciones significativas encontradas:

```
    POBREZA DEL HOGAR
         │
         ├──→ Dependencia del sector oficial (mayor matrícula oficial)
         │         └──→ Mayor hacinamiento por sede
         │
         ├──→ Peor transporte / mayor tiempo de viaje
         │         └──→ Fatiga, tardanzas, inasistencia
         │
         ├──→ Mayor violencia intrafamiliar
         │         └──→ Trauma, inestabilidad, bajo rendimiento
         │
         ├──→ Barreras económicas directas (costos, trabajo)
         │
         └──→ BAJO RENDIMIENTO → REPROBACIÓN
                                       │
                                       └──→ DESERCIÓN
```

### Hallazgo clave
La correlación directa pobreza→deserción NO es la más fuerte. El efecto es **indirecto**, mediado por la reprobación. La pobreza deteriora el rendimiento (por transporte, violencia, barreras), y el bajo rendimiento lleva a la deserción.

## 4.5 Conclusiones finales

In [5]:
conclusiones = """
CONCLUSIONES FINALES
====================
DataJam Edición 4 — Pobreza, Transporte y Educación en Bogotá

1. HALLAZGO PRINCIPAL:
   La pobreza NO causa directamente la deserción, sino que actúa a través
   de una cadena: Pobreza → Bajo rendimiento → Reprobación → Deserción.

2. EL TRANSPORTE ES BARRERA:
   Las localidades periféricas pobres (Ciudad Bolívar, Usme, Bosa) tienen
   tiempos de viaje al colegio de 40-48 min vs 25-33 min en zonas centrales.
   El transporte actúa como barrera doble: por COSTO y por TIEMPO.

3. LA REPROBACIÓN ES PREDICTOR DE DESERCIÓN:
   La correlación Reprobación→Deserción es la más fuerte del análisis.
   Intervenir en reprobación (tutorías, apoyo) es más eficiente que
   intervenir directamente en deserción.

4. VIOLENCIA INTRAFAMILIAR COMO CONTEXTO:
   Las localidades con mayor tasa de VIF en menores tienden a mayor
   reprobación. Ciudad Bolívar, Kennedy y Bosa concentran los casos.

5. RECOMENDACIONES:
   a) Focalizar intervención en REPROBACIÓN como indicador temprano
   b) Subsidiar transporte escolar en UPLs periféricas
   c) Reducir hacinamiento (>1000 est/sede) en localidades pobres
   d) Evaluar si programas de alimentación reducen deserción
"""

print(conclusiones)

# Guardar
with open(OUTPUT_DIR / 'CONCLUSIONES_FINALES.txt', 'w', encoding='utf-8') as f:
    f.write(conclusiones)
print('\n✓ Guardado: output/CONCLUSIONES_FINALES.txt')


CONCLUSIONES FINALES
DataJam Edición 4 — Pobreza, Transporte y Educación en Bogotá

1. HALLAZGO PRINCIPAL:
   La pobreza NO causa directamente la deserción, sino que actúa a través
   de una cadena: Pobreza → Bajo rendimiento → Reprobación → Deserción.

2. EL TRANSPORTE ES BARRERA:
   Las localidades periféricas pobres (Ciudad Bolívar, Usme, Bosa) tienen
   tiempos de viaje al colegio de 40-48 min vs 25-33 min en zonas centrales.
   El transporte actúa como barrera doble: por COSTO y por TIEMPO.

3. LA REPROBACIÓN ES PREDICTOR DE DESERCIÓN:
   La correlación Reprobación→Deserción es la más fuerte del análisis.
   Intervenir en reprobación (tutorías, apoyo) es más eficiente que
   intervenir directamente en deserción.

4. VIOLENCIA INTRAFAMILIAR COMO CONTEXTO:
   Las localidades con mayor tasa de VIF en menores tienden a mayor
   reprobación. Ciudad Bolívar, Kennedy y Bosa concentran los casos.

5. RECOMENDACIONES:
   a) Focalizar intervención en REPROBACIÓN como indicador temprano
   

## Ejecución del análisis completo

Para ejecutar todo el pipeline de una vez (sin notebooks), puedes usar:

```bash
# 1. Descargar datos
python scripts/descargar_datos.py

# 2. Ejecutar análisis completo
python analisis_final.py
```

Los resultados se guardan en la carpeta `output/`.